# Phase 2: Evaluate with Custom Tasks from Git

This notebook demonstrates the **`customTasks`** approach — pulling task definitions directly from the [lm-evaluation-harness](https://github.com/EleutherAI/lm-evaluation-harness) repository via Git.

## Why use this approach?

| Phase 1 (Built-in) | Phase 2 (Custom Git) |
|---|---|
| Limited to TrustyAI Tier 1/2 tasks | Access to **all** lm-evaluation-harness tasks |
| Only `kmmlu_direct_law` available | Full KMMLU (45 subjects), CLIcK (11 categories) |
| No extra configuration | Requires `customTasks.source.git` config |

## Available Korean Task Paths

In the lm-evaluation-harness repo, Korean tasks live under:

| Task Group | Git Path | Task Names |
|------------|----------|------------|
| KMMLU | `lm_eval/tasks/kmmlu` | `kmmlu`, `kmmlu_direct_*`, `kmmlu_hard_*` |
| CLIcK | `lm_eval/tasks/click` | `click`, `click_lang_*`, `click_cul_*` |
| KoBEST | `lm_eval/tasks/kobest` | `kobest_wic`, `kobest_copa`, etc. |

## Step 1: Configuration

In [ ]:
NAMESPACE = "hyo-project"
MODEL_NAME = "vllm-gemma4-e2b"
TOKENIZER = "google/gemma-2b"
BASE_URL = f"https://{MODEL_NAME}-predictor.{NAMESPACE}.svc.cluster.local:8443/v1/completions"
LIMIT = 5

# Custom task configuration
GIT_URL = "https://github.com/EleutherAI/lm-evaluation-harness.git"
GIT_BRANCH = "main"
TASK_PATH = "lm_eval/tasks/kmmlu"     # Change to lm_eval/tasks/click for CLIcK
TASK_NAME = "kmmlu_direct_law"         # Specific sub-task to run
JOB_NAME = "eval-custom-kmmlu"

print(f"Git source: {GIT_URL} @ {GIT_BRANCH}")
print(f"Task path: {TASK_PATH}")
print(f"Task name: {TASK_NAME}")

## Step 2: Review the YAML Template

Notice the `customTasks` section — this tells the operator to clone the Git repo and load tasks from the specified path:

In [ ]:
!cat samples/eval-kmmlu.yaml

## Step 3: Generate YAML for KMMLU

In [ ]:
yaml_content = f"""apiVersion: trustyai.opendatahub.io/v1alpha1
kind: LMEvalJob
metadata:
  name: {JOB_NAME}
spec:
  allowOnline: true
  model: local-completions
  modelArgs:
    - name: base_url
      value: "{BASE_URL}"
    - name: model
      value: "{MODEL_NAME}"
    - name: tokenizer
      value: "{TOKENIZER}"
    - name: verify_certificate
      value: "False"
    - name: num_concurrent
      value: "1"
    - name: max_retries
      value: "3"
    - name: tokenized_requests
      value: "false"
  taskList:
    customTasks:
      source:
        git:
          url: {GIT_URL}
          branch: {GIT_BRANCH}
          path: {TASK_PATH}
    taskNames:
      - {TASK_NAME}
  logSamples: true
  limit: "{LIMIT}"
  pod:
    container:
      env:
        - name: HF_TOKEN
          valueFrom:
            secretKeyRef:
              name: hf-token
              key: HF_TOKEN
        - name: OPENAI_API_KEY
          valueFrom:
            secretKeyRef:
              name: lmeval-sa-token
              key: token
"""

with open("/tmp/eval-custom.yaml", "w") as f:
    f.write(yaml_content)

print(yaml_content)

## Step 4: Submit the Evaluation Job

In [ ]:
!oc apply -f /tmp/eval-custom.yaml -n {NAMESPACE}

## Step 5: Monitor Execution

In [ ]:
import time

for i in range(12):
    !oc get pods -n {NAMESPACE} | grep {JOB_NAME}
    time.sleep(10)

## Step 6: Get Results

In [ ]:
!oc logs {JOB_NAME} -c main -n {NAMESPACE} 2>&1 | grep -A 5 "Tasks\|Metric"

In [ ]:
import json
import subprocess

result = subprocess.run(
    ["oc", "get", "lmevaljob", JOB_NAME, "-n", NAMESPACE,
     "-o", "jsonpath={.status.results}"],
    capture_output=True, text=True
)

if result.stdout:
    results = json.loads(result.stdout)
    print(json.dumps(results.get("results", {}), indent=2))
else:
    print("Results not yet available. Check pod status.")

## Step 7: Cleanup KMMLU Job

In [ ]:
!oc delete lmevaljob {JOB_NAME} -n {NAMESPACE}

---

## Bonus: Run CLIcK Benchmark

To run CLIcK instead of KMMLU, simply change the `TASK_PATH` and `TASK_NAME`:

In [ ]:
# CLIcK configuration
TASK_PATH_CLICK = "lm_eval/tasks/click"
TASK_NAME_CLICK = "click"               # Runs all 11 CLIcK categories
JOB_NAME_CLICK = "eval-custom-click"

yaml_click = f"""apiVersion: trustyai.opendatahub.io/v1alpha1
kind: LMEvalJob
metadata:
  name: {JOB_NAME_CLICK}
spec:
  allowOnline: true
  model: local-completions
  modelArgs:
    - name: base_url
      value: "{BASE_URL}"
    - name: model
      value: "{MODEL_NAME}"
    - name: tokenizer
      value: "{TOKENIZER}"
    - name: verify_certificate
      value: "False"
    - name: num_concurrent
      value: "1"
    - name: max_retries
      value: "3"
    - name: tokenized_requests
      value: "false"
  taskList:
    customTasks:
      source:
        git:
          url: {GIT_URL}
          branch: {GIT_BRANCH}
          path: {TASK_PATH_CLICK}
    taskNames:
      - {TASK_NAME_CLICK}
  logSamples: true
  limit: "{LIMIT}"
  pod:
    container:
      env:
        - name: HF_TOKEN
          valueFrom:
            secretKeyRef:
              name: hf-token
              key: HF_TOKEN
        - name: OPENAI_API_KEY
          valueFrom:
            secretKeyRef:
              name: lmeval-sa-token
              key: token
"""

with open("/tmp/eval-click.yaml", "w") as f:
    f.write(yaml_click)

print("CLIcK YAML generated. Run the next cell to apply.")

In [ ]:
# Uncomment to run CLIcK evaluation:
# !oc apply -f /tmp/eval-click.yaml -n {NAMESPACE}

## Summary

**When to use Phase 2 (Custom Tasks from Git):**

- Full benchmark suites (all 45 KMMLU subjects, all 11 CLIcK categories)
- Tasks not included in TrustyAI's built-in Tier 1/2 list
- Community-contributed or custom evaluation tasks
- Reproducible evaluations pinned to a specific commit

**Tips:**

- Use `commit` field instead of `branch` for reproducible results
- Set `limit` to a small number for testing, remove for full evaluations
- Running full benchmarks (1000+ samples) may take 10-30 minutes depending on model speed

**Results:** Accumulated evaluation results are tracked at [evaluate-llm-on-korean-dataset](https://github.com/hyogrin/evaluate-llm-on-korean-dataset)